# This notebook is to practice pooling techniques

##### Difficulty Levels:

Easy: AvgPool2d, MaxPool1d, AvgPool1d (similar to MaxPool2d you already did)
Medium: MaxPool2d with Stride (handle stride calculation)
Challenging: AdaptiveAvgPool2d (calculate dynamic regions)
Advanced: Conv2d (understand convolution operation, handle weights)

##### Key Concepts:
Pooling vs Convolution:

Pooling: No learnable parameters, just aggregation (max/mean)
Convolution: Learnable filters (weights), performs pattern detection

##### Stride:
python
stride=1: move 1 pixel at a time
stride=2: move 2 pixels (skip every other position)
Output size: (H - K) // stride + 1

##### 1D vs 2D:
1D: One spatial dimension (sequences) - shape: (B, C, L)
2D: Two spatial dimensions (images) - shape: (B, C, H, W)

##### Adaptive Pooling:
Specify output size instead of kernel size
Kernel automatically adapts to input size
Useful for handling variable-sized inputs


In [3]:
import torch
import torch.nn as nn

In [ ]:
# stride=1: move 1 pixel at a time
# stride=2: move 2 pixels (skip every other position)
# Output size: (H - K) // stride + 1

### MaxPool2d

https://docs.pytorch.org/docs/stable/generated/torch.nn.MaxPool2d.html

In [9]:
class MaxPool2d(nn.Module):
    """
    This class implements the MaxPool2d with a stride of 1. The
    implementation should be done with two loops for the spatial
    dimensions, being the rest of the operations vectorized.

    Attributes:
        kernel_size: Kernel size for the operation.
    """

    # Define attributes
    kernel_size: int

    def __init__(self, kernel_size: int) -> None:
        """
        This method is the constructor of the class.

        Args:
            kernel_size: Kernel size for the operation.

        Returns:
            None.
        """

        # Call super class constructor
        super().__init__()

        # Set attributes
        self.kernel_size = kernel_size

        return None

    def forward(self, inputs: torch.Tensor) -> torch.Tensor:
        """
        This method is the forward pass of the layer.

        Args:
            inputs: Inputs tensor. Dimensions: [batch, channels,
                height, width].

        Returns:
            Outputs tensor. Dimensions: [batch, channels,
                height - kernel_size + 1, width - kernel_size + 1].
        """

        # TODO
        B, C, H, W = inputs.shape
        K = self.kernel_size
        H_out, W_out = H - K + 1, W - K + 1
        
        outputs = torch.zeros([B, C, H_out, W_out], dtype=float)
        
        for i in range(H_out):
            for j in range(W_out):
                window = inputs[:, :, i:i+K, j:j+K].reshape(B, C, -1)
                outputs[:, :, i, j] = torch.max(window, dim=2).values
                
        return outputs

### AvgPool2d - AveragePooling

https://docs.pytorch.org/docs/stable/generated/torch.nn.AvgPool2d.html

In [6]:
# ============================================================================
# PRACTICE PROBLEM 1: AvgPool2d (stride=1)
# ============================================================================

class AvgPool2d(nn.Module):
    """
    Implements AvgPool2d with stride of 1.
    
    Takes the average of values in each kernel_size × kernel_size window.
    Use two loops for spatial dimensions, vectorize the rest.
    """
    
    kernel_size: int
    
    def __init__(self, kernel_size: int) -> None:
        """
        Args:
            kernel_size: Kernel size for the operation.
        
        Returns:
            None.
        """
        super().__init__()
        self.kernel_size = kernel_size
        return None
    
    def forward(self, inputs: torch.Tensor) -> torch.Tensor:
        """
        Args:
            inputs: Input tensor. Dimensions: [batch, channels, height, width].
        
        Returns:
            Output tensor. Dimensions: [batch, channels, 
                height - kernel_size + 1, width - kernel_size + 1].
        
        Hint:
        1. Extract dimensions: B, C, H, W = inputs.shape
        2. Calculate output size: H_out = H - K + 1, W_out = W - K + 1
        3. Create output tensor: torch.zeros(B, C, H_out, W_out)
        4. Loop over i (height) and j (width)
        5. Extract window: inputs[:, :, i:i+K, j:j+K]
        6. Reshape to (B, C, K*K) and take mean along dim=2
        7. Assign to outputs[:, :, i, j]
        """
        # TODO
        B, C, H, W = inputs.shape
        K = self.kernel_size
        H_out, W_out = H - K + 1, W - K + 1
        
        
        torch.zeros_like(B, C, H_out, W_out)
        
        for i in range(H_out):
            for j in range(W_out):
                window = inputs[:, :, i:i+K, j:j+K]
                outputs = window.reshape(B,C,-1).mean(dim=2)
                
        return outputs
                
        
        


# Test your implementation
def test_avgpool2d():
    # Create simple test input
    x = torch.tensor([[[[1., 2., 3.],
                        [4., 5., 6.],
                        [7., 8., 9.]]]])  # Shape: (1, 1, 3, 3)
    
    my_pool = AvgPool2d(kernel_size=2)
    pytorch_pool = nn.AvgPool2d(kernel_size=2, stride=1)
    
    result = my_pool(x)
    expected = pytorch_pool(x)
    
    print(f"Input shape: {x.shape}")
    print(f"Input:\n{x[0, 0]}\n")
    print(f"Your result shape: {result.shape}")
    print(f"Your result:\n{result[0, 0]}")
    print(f"Expected:\n{expected[0, 0]}")
    print(f"Match: {torch.allclose(result, expected)}\n")

In [8]:
test_avgpool2d()

Input shape: torch.Size([1, 1, 3, 3])
Input:
tensor([[1., 2., 3.],
        [4., 5., 6.],
        [7., 8., 9.]])

Your result shape: torch.Size([1, 1, 2, 2])
Your result:
tensor([[3., 4.],
        [6., 7.]])
Expected:
tensor([[3., 4.],
        [6., 7.]])
Match: True



# MaxPool2d with Stride

In [18]:
# ============================================================================
# PRACTICE PROBLEM 2: MaxPool2d with Stride
# ============================================================================

class MaxPool2dStride(nn.Module):
    """
    Implements MaxPool2d with configurable stride.
    
    Stride controls how much we move the window each step.
    stride=1: every position (like before)
    stride=2: skip every other position
    """
    
    kernel_size: int
    stride: int
    
    def __init__(self, kernel_size: int, stride: int = None) -> None:
        """
        Args:
            kernel_size: Kernel size for the operation.
            stride: Stride for the operation. If None, defaults to kernel_size.
        
        Returns:
            None.
        """
        super().__init__()
        self.kernel_size = kernel_size
        self.stride = stride if stride is not None else kernel_size
        return None
    
    def forward(self, inputs: torch.Tensor) -> torch.Tensor:
        """
        Args:
            inputs: Input tensor. Dimensions: [batch, channels, height, width].
        
        Returns:
            Output tensor. Dimensions: [batch, channels, H_out, W_out].
            where H_out = (H - K) // stride + 1
                  W_out = (W - K) // stride + 1
        
        Hint:
        1. Extract dimensions: B, C, H, W = inputs.shape
        2. Calculate output size: 
           H_out = (H - K) // stride + 1
           W_out = (W - K) // stride + 1
        3. Loop with stride: for i in range(0, H - K + 1, stride)
        4. Map loop index to output index: i_out = i // stride
        5. Extract window at inputs[:, :, i:i+K, j:j+K]
        6. Take max and assign to outputs[:, :, i_out, j_out]
        """
        # TODO
        B, C, H, W = inputs.shape
        K = self.kernel_size
        s = self.stride
        
        H_out = (H - K) // s + 1
        W_out = (W - K) // s + 1
        
        output = torch.zeros([B, C, H_out, W_out], dtype=torch.float)
        for i_out, i in enumerate(range(0, H - K + 1, s)):
            for j_out, j in enumerate(range(0, W - K + 1, s)):
                window = inputs[:,:,i:i+K, j:j+K].reshape(B, C, -1)
                output[:,:,i_out, j_out]=torch.max(window, dim=2).values
        return output


# Test your implementation
def test_maxpool2d_stride():
    x = torch.tensor([[[[1., 2., 3., 4.],
                        [5., 6., 7., 8.],
                        [9., 10., 11., 12.],
                        [13., 14., 15., 16.]]]])  # Shape: (1, 1, 4, 4)
    
    my_pool = MaxPool2dStride(kernel_size=2, stride=2)
    pytorch_pool = nn.MaxPool2d(kernel_size=2, stride=2)
    
    result = my_pool(x)
    expected = pytorch_pool(x)
    
    print(f"Input shape: {x.shape}")
    print(f"Input:\n{x[0, 0]}\n")
    print(f"Your result shape: {result.shape}")
    print(f"Your result:\n{result[0, 0]}")
    print(f"Expected:\n{expected[0, 0]}")
    print(f"Match: {torch.allclose(result, expected)}\n")


In [19]:
test_maxpool2d_stride()

Input shape: torch.Size([1, 1, 4, 4])
Input:
tensor([[ 1.,  2.,  3.,  4.],
        [ 5.,  6.,  7.,  8.],
        [ 9., 10., 11., 12.],
        [13., 14., 15., 16.]])

Your result shape: torch.Size([1, 1, 2, 2])
Your result:
tensor([[ 6.,  8.],
        [14., 16.]])
Expected:
tensor([[ 6.,  8.],
        [14., 16.]])
Match: True



### MaxPool1d (stride=1)

In [21]:
# ============================================================================
# PRACTICE PROBLEM 3: MaxPool1d (stride=1)
# ============================================================================

class MaxPool1d(nn.Module):
    """
    Implements MaxPool1d with stride of 1.
    
    1D version: operates on sequences (batch, channels, length).
    Common in time series and NLP.
    """
    
    kernel_size: int
    
    def __init__(self, kernel_size: int) -> None:
        """
        Args:
            kernel_size: Kernel size for the operation.
        
        Returns:
            None.
        """
        super().__init__()
        self.kernel_size = kernel_size
        return None
    
    def forward(self, inputs: torch.Tensor) -> torch.Tensor:
        """
        Args:
            inputs: Input tensor. Dimensions: [batch, channels, length].
        
        Returns:
            Output tensor. Dimensions: [batch, channels, length - kernel_size + 1].
        
        Hint:
        1. Extract dimensions: B, C, L = inputs.shape
        2. Calculate output length: L_out = L - K + 1
        3. Create output tensor: torch.zeros(B, C, L_out)
        4. Loop over i (only one spatial dimension!)
        5. Extract window: inputs[:, :, i:i+K]
        6. Reshape to (B, C, K) and take max along dim=2
        7. Assign to outputs[:, :, i]
        """
        # TODO
        B, C, L = inputs.shape
        k = self.kernel_size
        L_out = L - k + 1
        
        outputs = torch.zeros([B, C, L_out], dtype=torch.float)
        
        for i in range(L_out):
            window = inputs[:, :, i:i+k].reshape(B, C, -1)
            
            outputs[:, :, i] = torch.max(window, dim=2).values
            
        return outputs
        
        
        
        


# Test your implementation
def test_maxpool1d():
    x = torch.tensor([[[1., 2., 3., 4., 5.]],
                      [[5., 4., 3., 2., 1.]]])  # Shape: (2, 1, 5)
    
    my_pool = MaxPool1d(kernel_size=2)
    pytorch_pool = nn.MaxPool1d(kernel_size=2, stride=1)
    
    result = my_pool(x)
    expected = pytorch_pool(x)
    
    print(f"Input shape: {x.shape}")
    print(f"Input:\n{x}\n")
    print(f"Your result shape: {result.shape}")
    print(f"Your result:\n{result}")
    print(f"Expected:\n{expected}")
    print(f"Match: {torch.allclose(result, expected)}\n")

In [22]:
test_maxpool1d()

Input shape: torch.Size([2, 1, 5])
Input:
tensor([[[1., 2., 3., 4., 5.]],

        [[5., 4., 3., 2., 1.]]])

Your result shape: torch.Size([2, 1, 4])
Your result:
tensor([[[2., 3., 4., 5.]],

        [[5., 4., 3., 2.]]])
Expected:
tensor([[[2., 3., 4., 5.]],

        [[5., 4., 3., 2.]]])
Match: True



# AvgPool1d (stride=1)

In [21]:
# ============================================================================
# PRACTICE PROBLEM 4: AvgPool1d (stride=1)
# ============================================================================

class AvgPool1d(nn.Module):
    """
    Implements AvgPool1d with stride of 1.
    
    1D average pooling for sequences.
    """
    
    kernel_size: int
    
    def __init__(self, kernel_size: int) -> None:
        """
        Args:
            kernel_size: Kernel size for the operation.
        
        Returns:
            None.
        """
        super().__init__()
        self.kernel_size = kernel_size
        return None
    
    def forward(self, inputs: torch.Tensor) -> torch.Tensor:
        """
        Args:
            inputs: Input tensor. Dimensions: [batch, channels, length].
        
        Returns:
            Output tensor. Dimensions: [batch, channels, length - kernel_size + 1].
        
        Hint: Same as MaxPool1d but use .mean() instead of .max()
        """
        # TODO
        B, C, L = inputs.shape
        k =  self.kernel_size
        L_out = L - k + 1
        
        outputs = torch.zeros(B, C, L_out)
        
        for i in range(L_out):
            window = inputs[:, :, i:i+k]
            outputs[:, :, i] = window.mean(dim=2)
        return outputs


# Test your implementation
def test_avgpool1d():
    x = torch.tensor([[[1., 2., 3., 4., 5.]],
                      [[5., 4., 3., 2., 1.]]])  # Shape: (2, 1, 5)
    
    my_pool = AvgPool1d(kernel_size=2)
    pytorch_pool = nn.AvgPool1d(kernel_size=2, stride=1)
    
    result = my_pool(x)
    expected = pytorch_pool(x)
    
    print(f"Input shape: {x.shape}")
    print(f"Input:\n{x}\n")
    print(f"Your result shape: {result.shape}")
    print(f"Your result:\n{result}")
    print(f"Expected:\n{expected}")
    print(f"Match: {torch.allclose(result, expected)}\n")

In [22]:
test_avgpool1d()

Input shape: torch.Size([2, 1, 5])
Input:
tensor([[[1., 2., 3., 4., 5.]],

        [[5., 4., 3., 2., 1.]]])

Your result shape: torch.Size([2, 1, 4])
Your result:
tensor([[[1.5000, 2.5000, 3.5000, 4.5000]],

        [[4.5000, 3.5000, 2.5000, 1.5000]]])
Expected:
tensor([[[1.5000, 2.5000, 3.5000, 4.5000]],

        [[4.5000, 3.5000, 2.5000, 1.5000]]])
Match: True



### AdaptiveAvgPool2d (CHALLENGING!)

In [4]:
# ============================================================================
# PRACTICE PROBLEM 5: AdaptiveAvgPool2d (CHALLENGING!)
# ============================================================================

class AdaptiveAvgPool2d(nn.Module):
    """
    Implements AdaptiveAvgPool2d.
    
    Unlike regular pooling, you specify OUTPUT size, not kernel size.
    The kernel and stride are automatically calculated.
    
    Commonly used: AdaptiveAvgPool2d((1, 1)) to reduce any size to 1×1
    """
    
    output_size: tuple
    
    def __init__(self, output_size: tuple) -> None:
        """
        Args:
            output_size: Desired output size (H_out, W_out).
                        Can be int (square) or tuple.
        
        Returns:
            None.
        """
        super().__init__()
        if isinstance(output_size, int):
            self.output_size = (output_size, output_size)
        else:
            self.output_size = output_size
        return None
    
    def forward(self, inputs: torch.Tensor) -> torch.Tensor:
        """
        Args:
            inputs: Input tensor. Dimensions: [batch, channels, height, width].
        
        Returns:
            Output tensor. Dimensions: [batch, channels, H_out, W_out].
        
        Hint:
        1. For each output position (i, j), calculate input region:
           start_h = int(i * H / H_out)
           end_h = int((i + 1) * H / H_out)
           start_w = int(j * W / W_out)
           end_w = int((j + 1) * W / W_out)
        2. Extract region: inputs[:, :, start_h:end_h, start_w:end_w]
        3. Take mean over spatial dimensions: .mean(dim=[2, 3])
        4. Assign to outputs[:, :, i, j]
        """
        # TODO
        B, C, H, W = inputs.shape
        H_out = self.output_size[0]
        W_out = self.output_size[1]
        
        outputs = torch.zeros([B, C, H_out, W_out])
        
        for i in range(H_out):
            for j in range(W_out):
                start_h = int(i * H / H_out)
                end_h = int((i + 1) * H / H_out)
                start_w = int(j * W / W_out)
                end_w = int((j + 1) * W / W_out)
                
                window = inputs[:, :, start_h:end_h, start_w:end_w]
                
                outputs[:, :, i, j] = window.mean(dim=(2,3))
                
        return outputs

# Test your implementation
def test_adaptive_avgpool2d():
    x = torch.randn(2, 3, 8, 8)  # Random input
    
    my_pool = AdaptiveAvgPool2d((1, 1))
    pytorch_pool = nn.AdaptiveAvgPool2d((1, 1))
    
    result = my_pool(x)
    expected = pytorch_pool(x)
    
    print(f"Input shape: {x.shape}")
    print(f"Your result shape: {result.shape}")
    print(f"Expected shape: {expected.shape}")
    print(f"Match: {torch.allclose(result, expected, atol=1e-5)}\n")
    
    # Test with different output size
    my_pool2 = AdaptiveAvgPool2d((2, 2))
    pytorch_pool2 = nn.AdaptiveAvgPool2d((2, 2))
    
    result2 = my_pool2(x)
    expected2 = pytorch_pool2(x)
    
    print(f"Output size (2, 2):")
    print(f"Your result shape: {result2.shape}")
    print(f"Match: {torch.allclose(result2, expected2, atol=1e-5)}\n")

In [5]:
test_adaptive_avgpool2d()

Input shape: torch.Size([2, 3, 8, 8])
Your result shape: torch.Size([2, 3, 1, 1])
Expected shape: torch.Size([2, 3, 1, 1])
Match: True

Output size (2, 2):
Your result shape: torch.Size([2, 3, 2, 2])
Match: True



### Conv2d (stride=1, no padding) - ADVANCED!

https://docs.pytorch.org/docs/stable/generated/torch.nn.Conv2d.html

In [29]:
# ============================================================================
# PRACTICE PROBLEM 6: Conv2d (stride=1, no padding) - ADVANCED!
# ============================================================================

class Conv2d(nn.Module):
    """
    Implements Conv2d with stride=1 and no padding.
    
    Performs 2D convolution: applies learnable filters to input.
    This is the core operation in CNNs!
    
    Use two loops for spatial dimensions, vectorize the rest.
    """
    
    in_channels: int
    out_channels: int
    kernel_size: int
    weight: torch.nn.Parameter
    bias: torch.nn.Parameter
    
    def __init__(self, in_channels: int, out_channels: int, 
                 kernel_size: int, bias: bool = True) -> None:
        """
        Args:
            in_channels: Number of input channels.
            out_channels: Number of output channels (number of filters).
            kernel_size: Size of the convolving kernel.
            bias: If True, adds a learnable bias.
        
        Returns:
            None.
        """
        super().__init__()
        self.in_channels = in_channels
        self.out_channels = out_channels
        self.kernel_size = kernel_size
        
        # Initialize weights: (out_channels, in_channels, kernel_size, kernel_size)
        self.weight = nn.Parameter(
            torch.randn(out_channels, in_channels, kernel_size, kernel_size)
        )
        
        # Initialize bias: (out_channels,)
        if bias:
            self.bias = nn.Parameter(torch.zeros(out_channels))
        else:
            self.bias = None
        
        return None
    
    def forward(self, inputs: torch.Tensor) -> torch.Tensor:
        """
        Args:
            inputs: Input tensor. Dimensions: [batch, in_channels, height, width].
        
        Returns:
            Output tensor. Dimensions: [batch, out_channels, H_out, W_out].
            where H_out = H - kernel_size + 1
                  W_out = W - kernel_size + 1
        
        Hint:
        1. Extract dimensions: B, C_in, H, W = inputs.shape
        2. Calculate output size: H_out = H - K + 1, W_out = W - K + 1
        3. Create output: torch.zeros(B, out_channels, H_out, W_out)
        4. Loop over spatial positions (i, j)
        5. Extract window: inputs[:, :, i:i+K, j:j+K]  # Shape: (B, C_in, K, K)
        6. Reshape window to (B, C_in * K * K)
        7. Reshape weight to (out_channels, C_in * K * K)
        8. Matrix multiply: output = window @ weight.T  # Shape: (B, out_channels)
        9. Add bias if exists
        10. Assign to outputs[:, :, i, j]
        
        Alternative (easier but less efficient):
        - Use element-wise multiplication and sum:
          For each output channel c_out:
              outputs[:, c_out, i, j] = (window * weight[c_out]).sum(dim=(1,2,3))
        """
        # TODO
        B, C, H, W = inputs.shape
        k = self.kernel_size
        H_out, W_out = H - k + 1, W - k + 1
        
        outputs = torch.zeros([B, self.out_channels, H_out, W_out], dtype= inputs.dtype)
        
        for i in range(H_out):
            for j in range(W_out):
                window = inputs[:,:,i:i+k,j:j+k].unsqueeze(1)
                
                prod = self.weight * window
                
                outputs[:,:,i,j] = prod.sum(dim=(2,3,4)) + self.bias
                
        return outputs
                

        


# Test your implementation
def test_conv2d():
    # Simple test with small dimensions
    x = torch.randn(2, 3, 5, 5)  # 2 images, 3 channels, 5×5
    
    my_conv = Conv2d(in_channels=3, out_channels=4, kernel_size=3)
    pytorch_conv = nn.Conv2d(in_channels=3, out_channels=4, 
                             kernel_size=3, stride=1, padding=0)
    
    # Copy weights to ensure same initialization
    pytorch_conv.weight.data = my_conv.weight.data.clone()
    pytorch_conv.bias.data = my_conv.bias.data.clone()
    
    result = my_conv(x)
    expected = pytorch_conv(x)
    
    print(f"Input shape: {x.shape}")
    print(f"Your result shape: {result.shape}")
    print(f"Expected shape: {expected.shape}")
    print(f"Match: {torch.allclose(result, expected, atol=1e-4)}\n")


In [30]:
test_conv2d()

Input shape: torch.Size([2, 3, 5, 5])
Your result shape: torch.Size([2, 4, 3, 3])
Expected shape: torch.Size([2, 4, 3, 3])
Match: True



In [ ]:
# ============================================================================
# BONUS: Run all tests
# ============================================================================

def run_all_tests():
    """Run all test functions"""
    print("="*70)
    print("RUNNING ALL POOLING/CONVOLUTION TESTS")
    print("="*70 + "\n")
    
    tests = [
        ("AvgPool2d (stride=1)", test_avgpool2d),
        ("MaxPool2d with Stride", test_maxpool2d_stride),
        ("MaxPool1d (stride=1)", test_maxpool1d),
        ("AvgPool1d (stride=1)", test_avgpool1d),
        ("AdaptiveAvgPool2d", test_adaptive_avgpool2d),
        ("Conv2d", test_conv2d),
    ]
    
    for name, test_func in tests:
        print(f"--- {name} ---")
        try:
            test_func()
        except Exception as e:
            print(f"❌ Error: {e}\n")
    
    print("="*70)
    print("TESTS COMPLETE")
    print("="*70)

# Uncomment to run all tests:
# run_all_tests()


# ============================================================================
# SOLUTIONS (Don't look until you've tried!)
# ============================================================================

# Scroll down for solutions...
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#

"""
SOLUTIONS:

# 1. AvgPool2d
def forward(self, inputs):
    B, C, H, W = inputs.shape
    K = self.kernel_size
    H_out, W_out = H - K + 1, W - K + 1
    
    outputs = torch.zeros(B, C, H_out, W_out)
    
    for i in range(H_out):
        for j in range(W_out):
            window = inputs[:, :, i:i+K, j:j+K]
            outputs[:, :, i, j] = window.reshape(B, C, -1).mean(dim=2)
    
    return outputs


# 2. MaxPool2d with Stride
def forward(self, inputs):
    B, C, H, W = inputs.shape
    K = self.kernel_size
    S = self.stride
    H_out = (H - K) // S + 1
    W_out = (W - K) // S + 1
    
    outputs = torch.zeros(B, C, H_out, W_out)
    
    for i_out, i in enumerate(range(0, H - K + 1, S)):
        for j_out, j in enumerate(range(0, W - K + 1, S)):
            window = inputs[:, :, i:i+K, j:j+K]
            outputs[:, :, i_out, j_out] = window.reshape(B, C, -1).max(dim=2).values
    
    return outputs


# 3. MaxPool1d
def forward(self, inputs):
    B, C, L = inputs.shape
    K = self.kernel_size
    L_out = L - K + 1
    
    outputs = torch.zeros(B, C, L_out)
    
    for i in range(L_out):
        window = inputs[:, :, i:i+K]
        outputs[:, :, i] = window.max(dim=2).values
    
    return outputs


# 4. AvgPool1d
def forward(self, inputs):
    B, C, L = inputs.shape
    K = self.kernel_size
    L_out = L - K + 1
    
    outputs = torch.zeros(B, C, L_out)
    
    for i in range(L_out):
        window = inputs[:, :, i:i+K]
        outputs[:, :, i] = window.mean(dim=2)
    
    return outputs


# 5. AdaptiveAvgPool2d
def forward(self, inputs):
    B, C, H, W = inputs.shape
    H_out, W_out = self.output_size
    
    outputs = torch.zeros(B, C, H_out, W_out)
    
    for i in range(H_out):
        for j in range(W_out):
            start_h = int(i * H / H_out)
            end_h = int((i + 1) * H / H_out)
            start_w = int(j * W / W_out)
            end_w = int((j + 1) * W / W_out)
            
            region = inputs[:, :, start_h:end_h, start_w:end_w]
            outputs[:, :, i, j] = region.mean(dim=[2, 3])
    
    return outputs


# 6. Conv2d (Simple version)
def forward(self, inputs):
    B, C_in, H, W = inputs.shape
    K = self.kernel_size
    H_out = H - K + 1
    W_out = W - K + 1
    
    outputs = torch.zeros(B, self.out_channels, H_out, W_out)
    
    for i in range(H_out):
        for j in range(W_out):
            # Extract window: (B, C_in, K, K)
            window = inputs[:, :, i:i+K, j:j+K]
            
            # For each output channel
            for c_out in range(self.out_channels):
                # Element-wise multiply and sum
                conv_result = (window * self.weight[c_out]).sum(dim=(1, 2, 3))
                
                # Add bias
                if self.bias is not None:
                    conv_result += self.bias[c_out]
                
                outputs[:, c_out, i, j] = conv_result
    
    return outputs


# 6. Conv2d (More efficient version using matrix multiplication)
def forward(self, inputs):
    B, C_in, H, W = inputs.shape
    K = self.kernel_size
    H_out = H - K + 1
    W_out = W - K + 1
    
    outputs = torch.zeros(B, self.out_channels, H_out, W_out)
    
    # Reshape weight: (out_channels, C_in * K * K)
    weight_reshaped = self.weight.reshape(self.out_channels, -1)
    
    for i in range(H_out):
        for j in range(W_out):
            # Extract and reshape window: (B, C_in * K * K)
            window = inputs[:, :, i:i+K, j:j+K].reshape(B, -1)
            
            # Matrix multiply: (B, C_in*K*K) @ (C_in*K*K, out_channels)
            conv_result = window @ weight_reshaped.T  # (B, out_channels)
            
            # Add bias
            if self.bias is not None:
                conv_result += self.bias
            
            outputs[:, :, i, j] = conv_result
    
    return outputs
"""

### Conv1d

https://docs.pytorch.org/docs/stable/generated/torch.nn.Conv2d.html